# Datamine EXPFLD Process Example

This notebook demonstrates how to configure and run the **`expfld`** process wrapper in `dmstudio`.

## Process Description

## EXPFLD

Copy a file, while expanding the number of records so that a record is output for given increments of a start field until an end field value is reached.

A typical use is where there are properties defined over a range of sample numbers, and these are to be combined with other data entered for each sample. For example, suppose the file contains the following fields:-

## BHID    FROMNO   TONO    DENSITY

where the **DENSITY** is defined for the range of sample numbers in **FROMNO** and **TONO**. EXPFLD could be used to expand this into a new file containing the fields:-

BHID Sample DENSITY

This file contains a record of the **DENSITY** for each sample number, and as such may be combined with other files in this format.

The records output are for values of * **START** , * **START** +@**INCRMENT** , * **START** +2*@**INCRMENT** , and so on, until the value of * **END** for the input record is reached. This means that @**INCRMENT** must never be zero, or an infinite number of records lie between the values.

### Input Files:

* **in** (Undefined):
  Input file containing numeric explicit fields **START** and **END** defining the range.
  Required=Yes

### Output Files:

* **out** (Undefined):
  Output file containing extra records between the given ranges, the actual value being
  held in field NEWFIELD.
  Required=Yes

### Fields:

* **start** (Numeric : IN):
  Name of field giving the start of the range.
  Default=Undefined
  Required=Yes

* **end** (Numeric : IN):
  Name of field giving the end of the range.
  Default=Undefined
  Required=Yes

* **newfield** (Numeric : OUT):
  Name of field in output file containing the value for the record within the range.
  Default=Undefined
  Required=Yes

### Parameters:

* **incrment**:
  Increment to be applied to **START** within range.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=Yes

In [ ]:
# Step 1: Connect to Datamine and Initialize Sandbox
import os
import shutil
import glob

import pandas as pd

from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Initialize active project sandbox using the shared test_sandbox project
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()
agent.initialize_sandbox(notebook_folder)

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('expfld')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine database locally to this sandbox folder using a `t_` prefix.

In [ ]:
# Step 3: Copy VBOP datasets dynamically from Database to test_sandbox
files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

agent.copy_database_files(files_to_copy)

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute expfld
print("Running expfld...")
dm_cmd.expfld(
    in_i='t_assays',  # required
    out_o='t_expfld_out',  # required
    start_f='optional',  # required
    end_f='optional',  # required
    newfield_f='optional',  # required
    incrment_p='required_val',  # required
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("expfld execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = 't_expfld_out.dmx'
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob("t_*.*")
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    if os.path.exists(f):
        try:
            os.remove(f)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = '__pycache__'
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")